In [1]:
import pandas as pd
import numpy as np
import pickle

from sklearn.preprocessing import (
    OneHotEncoder,
    MinMaxScaler,
    QuantileTransformer
)

from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.pipeline import Pipeline

from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier

from sklearn.ensemble import (
    RandomForestClassifier,
    GradientBoostingClassifier,
    AdaBoostClassifier
)

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

from sklearn.naive_bayes import GaussianNB, BernoulliNB
from sklearn.metrics import accuracy_score

In [2]:
df = pd.read_csv("./data/titanic_clean.csv")

print(f"Filas: {df.shape[0]}")
print(f"Columnas: {df.shape[1]}")
print(f"Valores nulos: {df.isnull().sum().sum()}")

df.head()

Filas: 891
Columnas: 8
Valores nulos: 0


,Survived,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked
0,0,3,male,22.0,1,0,7.2500,S
1,1,1,female,38.0,1,0,71.2833,C
2,1,3,female,26.0,0,0,7.9250,S
3,1,1,female,35.0,1,0,53.1000,S
4,0,3,male,35.0,0,0,8.0500,S


In [3]:
X = df.drop(columns=["Survived"])
y = df["Survived"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=100,
    stratify=y
)

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("y_train:", y_train.shape)
print("y_test:", y_test.shape)

X_train.head()

X_train: (712, 7)
X_test: (179, 7)
y_train: (712,)
y_test: (179,)


,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked
96,1,male,71.000000,0,0,34.6542,C
220,3,male,16.000000,0,0,8.0500,S
870,3,male,26.000000,0,0,7.8958,S
798,3,male,30.000000,0,0,7.2292,C
837,3,male,29.699118,0,0,8.0500,S


In [4]:
preprocessor = ColumnTransformer(
    transformers=[
        (
            "onehot",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=False
            ),
            ["Sex", "Embarked"]
        ),
        (
            "age",
            QuantileTransformer(
                output_distribution="normal",
                n_quantiles=500,
                random_state=42
            ),
            ["Age"]
        ),
        (
            "fare",
            QuantileTransformer(
                output_distribution="normal",
                n_quantiles=500,
                random_state=42
            ),
            ["Fare"]
        )
    ],
    remainder="passthrough"
)

preprocessor

,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('onehot', ...), ('age', ...), ...]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'passthrough'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers contains sparse matrices,these will be stacked as a sparse matrix if the overall density islower than this value. Use ``sparse_threshold=0`` to always returndense. When the transformed output consists of all dense data, thestacked result will be dense, and this keyword will be ignored.",0.3
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details.",None
,"transformer_weights transformer_weights: dict, default=NoneMultiplicative weights for features per transformer. The output of thetransformer is multiplied by these weights. Keys are transformer names,values the weights.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each transformer will beprinted as it is completed.",False
,"verbose_feature_names_out verbose_feature_names_out: bool, str or Callable[[str, str], str], default=True- If True, :meth:`ColumnTransformer.get_feature_names_out` will prefix all feature names with the name of the transformer that generated that feature. It is equivalent to setting `verbose_feature_names_out=""{transformer_name}__{feature_name}""`.- If False, :meth:`ColumnTransformer.get_feature_names_out` will not prefix any feature names and will error if feature names are not unique.- If ``Callable[[str, str], str]``, :meth:`ColumnTransformer.get_feature_names_out` will rename all the features using the name of the transformer. The first argument of the callable is the transformer name and the second argument is the feature name. The returned string will be the new feature name.- If ``str``, it must be a string ready for formatting. The given string will be formatted using two field names: ``transformer_name`` and `

In [8]:
modelos = {
    "Regresión Logística": {
        "modelo": LogisticRegression(
            random_state=42,
            solver="saga"
        ),
        "parametros": {
            "model__C": [0.01, 0.1, 1, 10],
            "model__l1_ratio": [0, 1],
            "model__max_iter": [500, 1000]
        }
    },

    "Clasificador de Vectores de Soporte": {
        "modelo": SVC(),
        "parametros": {
            "model__kernel": ["linear", "poly", "rbf", "sigmoid"],
            "model__C": [0.1, 1, 10]
        }
    },

    "Clasificador de Árbol de Decisión": {
        "modelo": DecisionTreeClassifier(random_state=42),
        "parametros": {
            "model__splitter": ["best", "random"],
            "model__max_depth": [None, 1, 2, 3, 4]
        }
    },

    "Clasificador de Bosques Aleatorios": {
        "modelo": RandomForestClassifier(random_state=42),
        "parametros": {
            "model__n_estimators": [10, 100],
            "model__max_depth": [None, 1, 2, 3, 4],
            "model__max_features": ["sqrt", "log2", None]
        }
    },

    "Clasificador de Gradient Boosting": {
        "modelo": GradientBoostingClassifier(random_state=42),
        "parametros": {
            "model__n_estimators": [10, 100],
            "model__max_depth": [None, 1, 2, 3, 4]
        }
    },

    "Clasificador AdaBoost": {
        "modelo": AdaBoostClassifier(random_state=42),
        "parametros": {
            "model__n_estimators": [10, 100]
        }
    },

    "Clasificador K-Nearest Neighbors": {
        "modelo": KNeighborsClassifier(),
        "parametros": {
            "model__n_neighbors": [3, 5, 7]
        }
    },

    "Clasificador XGBoost": {
        "modelo": XGBClassifier(
            random_state=42,
            eval_metric="logloss"
        ),
        "parametros": {
            "model__n_estimators": [10, 100],
            "model__max_depth": [1, 2, 3, 6]
        }
    },

    "Clasificador LGBM": {
        "modelo": LGBMClassifier(
            random_state=42,
            verbosity=-1
        ),
        "parametros": {
            "model__n_estimators": [10, 100],
            "model__max_depth": [-1, 1, 2, 3],
            "model__learning_rate": [0.1, 0.2, 0.3]
        }
    },

    "GaussianNB": {
        "modelo": GaussianNB(),
        "parametros": {}
    },

    "Clasificador Naive Bayes": {
        "modelo": BernoulliNB(),
        "parametros": {
            "model__alpha": [0.1, 1.0, 10.0]
        }
    }
}

In [9]:
print(f"Modelos configurados: {len(modelos)}")

for nombre in modelos:
    print("-", nombre)

Modelos configurados: 11
- Regresión Logística
- Clasificador de Vectores de Soporte
- Clasificador de Árbol de Decisión
- Clasificador de Bosques Aleatorios
- Clasificador de Gradient Boosting
- Clasificador AdaBoost
- Clasificador K-Nearest Neighbors
- Clasificador XGBoost
- Clasificador LGBM
- GaussianNB
- Clasificador Naive Bayes


In [10]:
puntajes_modelos = []
mejor_precision = 0
mejor_estimador = None
mejor_modelo = None
estimadores = {}

for nombre, info_modelo in modelos.items():
    print(f"Entrenando: {nombre}")

    pipeline = Pipeline(
        steps=[
            ("preprocessor", preprocessor),
            ("scaler", MinMaxScaler()),
            ("model", info_modelo["modelo"])
        ]
    )

    grid_search = GridSearchCV(
        estimator=pipeline,
        param_grid=info_modelo["parametros"],
        cv=5,
        scoring="accuracy",
        verbose=0,
        n_jobs=-1
    )

    grid_search.fit(X_train, y_train)

    y_pred = grid_search.predict(X_test)
    precision = accuracy_score(y_test, y_pred)

    puntajes_modelos.append({
        "Modelo": nombre,
        "Precisión": precision,
        "Mejores hiperparámetros": grid_search.best_params_
    })

    estimadores[nombre] = grid_search.best_estimator_

    if precision > mejor_precision:
        mejor_modelo = nombre
        mejor_precision = precision
        mejor_estimador = grid_search.best_estimator_

    print(f"Precisión: {precision:.4f}")

Entrenando: Regresión Logística
Precisión: 0.8324
Entrenando: Clasificador de Vectores de Soporte
Precisión: 0.8380
Entrenando: Clasificador de Árbol de Decisión
Precisión: 0.8324
Entrenando: Clasificador de Bosques Aleatorios
Precisión: 0.8492
Entrenando: Clasificador de Gradient Boosting
Precisión: 0.8436
Entrenando: Clasificador AdaBoost
Precisión: 0.7933
Entrenando: Clasificador K-Nearest Neighbors
Precisión: 0.7877
Entrenando: Clasificador XGBoost
Precisión: 0.8547
Entrenando: Clasificador LGBM
Precisión: 0.8380
Entrenando: GaussianNB
Precisión: 0.7933
Entrenando: Clasificador Naive Bayes
Precisión: 0.8101


In [11]:
metricas = (
    pd.DataFrame(puntajes_modelos)
    .sort_values("Precisión", ascending=False)
    .reset_index(drop=True)
)

display(metricas)

print("---------------------------------------------")
print("MEJOR PIPELINE DE CLASIFICACIÓN")
print(f"Modelo: {mejor_modelo}")
print(f"Precisión: {mejor_precision:.4f}")
print(f"Estimador: {mejor_estimador}")

,Modelo,Precisión,Mejores hiperparámetros
0,Clasificador XGBoost,0.854749,"{'model__max_depth': 3, 'model__n_estimators':..."
1,Clasificador de Bosques Aleatorios,0.849162,"{'model__max_depth': 4, 'model__max_features':..."
2,Clasificador de Gradient Boosting,0.843575,"{'model__max_depth': 3, 'model__n_estimators':..."
3,Clasificador de Vectores de Soporte,0.837989,"{'model__C': 1, 'model__kernel': 'poly'}"
4,Clasificador LGBM,0.837989,"{'model__learning_rate': 0.1, 'model__max_dept..."
5,Clasificador de Árbol de Decisión,0.832402,"{'model__max_depth': 3, 'model__splitter': 'be..."
6,Regresión Logística,0.832402,"{'model__C': 0.01, 'model__l1_ratio': 0, 'mode..."
7,Clasificador Naive Bayes,0.810056,{'model__alpha': 0.1}
8,Clasificador AdaBoost,0.793296,{'model__n_estimators': 100}
9,GaussianNB,0.793296,{}


---------------------------------------------
MEJOR PIPELINE DE CLASIFICACIÓN
Modelo: Clasificador XGBoost
Precisión: 0.8547
Estimador: Pipeline(steps=[('preprocessor',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('onehot',
                                                  OneHotEncoder(handle_unknown='ignore',
                                                                sparse_output=False),
                                                  ['Sex', 'Embarked']),
                                                 ('age',
                                                  QuantileTransformer(n_quantiles=500,
                                                                      output_distribution='normal',
                                                                      random_state=42),
                                                  ['Age']),
                                                 ('fare',
              

In [12]:
nuevo_pasajero = pd.DataFrame([{
    "Pclass": 2,
    "Sex": "male",
    "Age": 46,
    "SibSp": 0,
    "Parch": 0,
    "Fare": 7.25,
    "Embarked": "C"
}])

prediccion = int(mejor_estimador.predict(nuevo_pasajero)[0])

resultado = "sobrevivió" if prediccion == 1 else "no sobrevivió"

print(f"Predicción: {prediccion}")
print(f"Resultado: {resultado}")

Predicción: 0
Resultado: no sobrevivió


In [13]:
with open("pipeline.pkl", "wb") as archivo:
    pickle.dump(mejor_estimador, archivo)

print("Pipeline guardado correctamente en pipeline.pkl")

Pipeline guardado correctamente en pipeline.pkl


In [14]:
with open("pipeline.pkl", "rb") as archivo:
    pipeline_cargado = pickle.load(archivo)

prediccion_prueba = pipeline_cargado.predict(nuevo_pasajero)

print("Predicción del pipeline cargado:", prediccion_prueba)

Predicción del pipeline cargado: [0]
